In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images/N/N_1466.jpg
/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images/N/N_892.jpg
/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images/N/N_1413.jpg
/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images/N/N_125.jpg
/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images/N/N_649.jpg
/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images/N/N_247.jpg
/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images/N/N_594.jpg
/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images/N/N_1424.jpg
/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images/N/N_1376.jpg
/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images/N/N_564.jpg
/kaggle/input/datasets/namyajain2810

In [3]:
# PART 1 — Load both folders fully, no caps
import os, cv2, numpy as np

original_path = "/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/original_images"
custom_path   = "/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images"

def load_images(path, limit=None):
    data, labels = [], []
    for label in sorted(os.listdir(path)):
        folder = os.path.join(path, label)
        if not os.path.isdir(folder):
            continue
        files = sorted(os.listdir(folder))
        if limit:
            files = files[:limit]
        for img_name in files:
            img = cv2.imread(os.path.join(folder, img_name))
            if img is None:
                continue
            img = cv2.resize(img, (64, 64))
            data.append(img)
            labels.append(label)
    return data, labels

orig_data, orig_labels = load_images(original_path, limit=300)
cust_data, cust_labels = load_images(custom_path)  # full 54,000, no limit

print("original_images:", len(orig_data))
print("custom_images:", len(cust_data))

original_images: 10800
custom_images: 54000


In [4]:
# PART 2 — Shared label encoder + normalize
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(sorted(set(orig_labels)))  # same 36 classes in both folders
print("Classes:", len(le.classes_))

X_orig = np.array(orig_data) / 255.0
y_orig = le.transform(orig_labels)

X_cust_full = np.array(cust_data) / 255.0
y_cust_full = le.transform(cust_labels)

print("X_orig:", X_orig.shape, "X_cust_full:", X_cust_full.shape)

Classes: 36
X_orig: (10800, 64, 64, 3) X_cust_full: (54000, 64, 64, 3)


In [ ]:
# PART 3 — EXPERIMENT A: train on original_images only, with augmentation + regularization
from sklearn.model_selection import train_test_split
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization,
    RandomRotation, RandomZoom, RandomBrightness, RandomTranslation
)
from tensorflow.keras.callbacks import EarlyStopping

X_train_a, X_val_a, y_train_a, y_val_a = train_test_split(
    X_orig, y_orig, test_size=0.2, stratify=y_orig, random_state=42
)

def build_model(num_classes):
    return Sequential([
        Input(shape=(64, 64, 3)),
        RandomRotation(0.08), RandomZoom(0.1), RandomTranslation(0.1, 0.1),
        RandomBrightness(0.15, value_range=(0, 1)),   # <-- the fix
        Conv2D(32, (3,3), activation='relu'), BatchNormalization(), MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation='relu'), BatchNormalization(), MaxPooling2D((2,2)),
        Flatten(), Dropout(0.3),
        Dense(128, activation='relu'), Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])

model_a = build_model(len(le.classes_))
model_a.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_a = model_a.fit(
    X_train_a, y_train_a, epochs=30, validation_data=(X_val_a, y_val_a),
    callbacks=[EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True)]
)

2026-06-26 16:18:39.402860: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782490719.429083    1428 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782490719.437744    1428 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782490719.458609    1428 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782490719.458637    1428 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782490719.458639    1428 computation_placer.cc:177] computation placer alr

Epoch 1/30


I0000 00:00:1782490736.929476    1477 cuda_dnn.cc:529] Loaded cuDNN version 91002


270/270 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.1000 - loss: 4.1242 - val_accuracy: 0.1185 - val_loss: 4.6690
Epoch 2/30
270/270 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.2737 - loss: 2.5926 - val_accuracy: 0.4995 - val_loss: 1.7188
Epoch 3/30
270/270 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.4254 - loss: 1.9103 - val_accuracy: 0.5306 - val_loss: 1.6400
Epoch 4/30
270/270 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.5321 - loss: 1.5125 - val_accuracy: 0.7032 - val_loss: 0.9443
Epoch 5/30
270/270 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.6099 - loss: 1.2424 - val_accuracy: 0.8560 - val_loss: 0.4556
Epoch 6/30
270/270 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.6471 - loss: 1.0859 - val_accuracy: 0.8324 - val_loss: 0.5167
Epoch 7/30
270/270 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.7104 - loss: 0.9268 - val_accuracy: 0.8852 - val_loss: 0.3249
Epoch 8/30
270/270 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.7237 - loss: 0.8681 - val_accuracy: 0.74

In [ ]:
# PART 4 — Zero-shot diagnostic: evaluate Experiment A on the FULL untouched custom_images
loss_a, acc_a = model_a.evaluate(X_cust_full, y_cust_full)
print("EXPERIMENT A — zero-shot accuracy on full custom_images:", acc_a)

with open("experiment_a_accuracy.txt", "w") as f:
    f.write(str(acc_a))
model_a.save("model_a_original_only.keras")

1688/1688 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.2761 - loss: 3.4569
EXPERIMENT A — zero-shot accuracy on full custom_images: 0.27235186100006104


In [ ]:
# PART 5 — EXPERIMENT B: mix in 80% of custom_images, hold out 20% as a true test set
idx_train, idx_test = train_test_split(
    np.arange(len(cust_data)), test_size=0.2, stratify=cust_labels, random_state=42
)
cust_train_imgs = [cust_data[i] for i in idx_train]; cust_train_lbls = [cust_labels[i] for i in idx_train]
cust_test_imgs  = [cust_data[i] for i in idx_test];  cust_test_lbls  = [cust_labels[i] for i in idx_test]

X_pool = np.array(orig_data + cust_train_imgs) / 255.0
y_pool = le.transform(orig_labels + cust_train_lbls)

X_test_b = np.array(cust_test_imgs) / 255.0
y_test_b = le.transform(cust_test_lbls)

X_train_b, X_val_b, y_train_b, y_val_b = train_test_split(
    X_pool, y_pool, test_size=0.15, stratify=y_pool, random_state=42
)
print("Train B:", X_train_b.shape, "Val B:", X_val_b.shape, "Held-out test B:", X_test_b.shape)

model_b = build_model(len(le.classes_))
model_b.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_b = model_b.fit(
    X_train_b, y_train_b, epochs=30, validation_data=(X_val_b, y_val_b),
    callbacks=[EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True)]
)

Train B: (45900, 64, 64, 3) Val B: (8100, 64, 64, 3) Held-out test B: (10800, 64, 64, 3)
Epoch 1/30
1435/1435 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - accuracy: 0.2142 - loss: 2.9729 - val_accuracy: 0.6317 - val_loss: 1.1324
Epoch 2/30
1435/1435 ━━━━━━━━━━━━━━━━━━━━ 22s 15ms/step - accuracy: 0.5534 - loss: 1.4199 - val_accuracy: 0.4958 - val_loss: 1.7976
Epoch 3/30
1435/1435 ━━━━━━━━━━━━━━━━━━━━ 22s 15ms/step - accuracy: 0.6772 - loss: 1.0172 - val_accuracy: 0.7454 - val_loss: 0.7865
Epoch 4/30
1435/1435 ━━━━━━━━━━━━━━━━━━━━ 22s 15ms/step - accuracy: 0.7416 - loss: 0.8138 - val_accuracy: 0.8014 - val_loss: 0.6386
Epoch 5/30
1435/1435 ━━━━━━━━━━━━━━━━━━━━ 22s 15ms/step - accuracy: 0.7828 - loss: 0.6597 - val_accuracy: 0.9009 - val_loss: 0.2903
Epoch 6/30
1435/1435 ━━━━━━━━━━━━━━━━━━━━ 22s 15ms/step - accuracy: 0.8136 - loss: 0.5788 - val_accuracy: 0.9328 - val_loss: 0.2039
Epoch 7/30
1435/1435 ━━━━━━━━━━━━━━━━━━━━ 21s 15ms/step - accuracy: 0.8329 - loss: 0.5017 - val_accuracy: 0.8472 - val_

In [ ]:
# PART 6 — Evaluate Experiment B on its true held-out test set, save as the model to ship
loss_b, acc_b = model_b.evaluate(X_test_b, y_test_b)
print("EXPERIMENT B — accuracy on held-out custom_images (never trained on):", acc_b)

with open("experiment_b_accuracy.txt", "w") as f:
    f.write(str(acc_b))

model_b.save("sign_model.keras")  # this is the one to carry into week 3

import pickle
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9986 - loss: 0.0104
EXPERIMENT B — accuracy on held-out custom_images (never trained on): 0.9984259009361267


In [ ]:
# PART 7 — Compare both experiments and save a results table for documentation
print("=== SUMMARY ===")
print(f"Experiment A (original_images only): {acc_a*100:.2f}% on full custom_images")
print(f"Experiment B (mixed training): {acc_b*100:.2f}% on held-out 20% custom_images")

import pandas as pd
pd.DataFrame({
    "experiment": ["A: original_images only", "B: mixed training"],
    "test_set": ["full custom_images (zero-shot)", "held-out 20% custom_images"],
    "accuracy": [acc_a, acc_b]
}).to_csv("experiment_comparison.csv", index=False)

=== SUMMARY ===
Experiment A (original_images only): 27.24% on full custom_images
Experiment B (mixed training): 99.84% on held-out 20% custom_images


In [ ]:
!pip install mediapipe==0.10.21 -q

# Kernel restart zaruri hai install ke baad
import os
os.kill(os.getpid(), 9)

In [1]:
import mediapipe as mp
import cv2
import os
import numpy as np

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.3
)

print("MediaPipe version:", mp.__version__)
print("Setup successful!")

2026-06-26 16:29:04.932103: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782491344.958674    2583 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782491344.967323    2583 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782491344.988562    2583 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782491344.988598    2583 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782491344.988601    2583 computation_placer.cc:177] computation placer alr

MediaPipe version: 0.10.21
Setup successful!


In [2]:
custom_path = "/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images"

print(f"{'Class':<10} {'Total':>8} {'Detected':>10} {'Skipped':>10} {'Skip%':>8}")
print("-" * 50)

total_all = 0
detected_all = 0

for label in sorted(os.listdir(custom_path)):
    folder = os.path.join(custom_path, label)
    if not os.path.isdir(folder):
        continue

    files = sorted(os.listdir(folder))[:200]
    total = 0
    detected = 0

    for img_name in files:
        img = cv2.imread(os.path.join(folder, img_name))
        if img is None:
            continue
        total += 1
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        result = hands.process(img_rgb)
        if result.multi_hand_landmarks:
            detected += 1

    skipped = total - detected
    skip_pct = (skipped / total * 100) if total > 0 else 0
    flag = " ⚠️" if skip_pct > 40 else ""
    print(f"{label:<10} {total:>8} {detected:>10} {skipped:>10} {skip_pct:>7.1f}%{flag}")

    total_all += total
    detected_all += detected

hands.close()

print("-" * 50)
overall_skip = ((total_all - detected_all) / total_all * 100)
print(f"{'TOTAL':<10} {total_all:>8} {detected_all:>10} {total_all-detected_all:>10} {overall_skip:>7.1f}%")
print(f"\nOverall detection rate: {detected_all/total_all*100:.1f}%")

Class         Total   Detected    Skipped    Skip%
--------------------------------------------------


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1782491353.142757    2627 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782491353.178679    2627 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782491353.191487    2627 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


0               200        200          0     0.0%
1               200         96        104    52.0% ⚠️
2               200        200          0     0.0%
3               200        200          0     0.0%
4               200        200          0     0.0%
5               200        200          0     0.0%
6               200        200          0     0.0%
7               200        200          0     0.0%
8               200        200          0     0.0%
9               200        200          0     0.0%
A               200        200          0     0.0%
B               200        200          0     0.0%
C               200        200          0     0.0%
D               200        200          0     0.0%
E               200        200          0     0.0%
F               200        200          0     0.0%
G               200        200          0     0.0%
H               200        200          0     0.0%
I               200        200          0     0.0%
J               200        2

In [3]:
# Sirf class "1" test karo lower confidence se
hands_low = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.1
)

folder = os.path.join(custom_path, "1")
files = sorted(os.listdir(folder))[:200]
detected = 0

for img_name in files:
    img = cv2.imread(os.path.join(folder, img_name))
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = hands_low.process(img_rgb)
    if result.multi_hand_landmarks:
        detected += 1

hands_low.close()
print(f"Class 1 with confidence=0.1: {detected}/200 detected ({detected/200*100:.1f}%)")

W0000 00:00:1782491631.082980    2633 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782491631.124906    2633 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Class 1 with confidence=0.1: 99/200 detected (49.5%)


In [4]:
import os, cv2, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization,
    RandomRotation, RandomZoom, RandomBrightness, RandomTranslation
)
from tensorflow.keras.callbacks import EarlyStopping

original_path = "/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/original_images"
custom_path   = "/kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images"

# ─── GREYSCALE load function ───────────────────────────────────────────────
def load_images_grey(path, limit=None):
    data, labels = [], []
    for label in sorted(os.listdir(path)):
        folder = os.path.join(path, label)
        if not os.path.isdir(folder):
            continue
        files = sorted(os.listdir(folder))
        if limit:
            files = files[:limit]
        for img_name in files:
            img = cv2.imread(os.path.join(folder, img_name))
            if img is None:
                continue
            img = cv2.resize(img, (64, 64))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)      # greyscale
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)      # 3 channel wapas
            data.append(img)
            labels.append(label)
    print(f"Loaded {len(data)} images from {path}")
    return data, labels

orig_data, orig_labels = load_images_grey(original_path, limit=300)
cust_data, cust_labels = load_images_grey(custom_path)

print("Loading complete.")

Loaded 10800 images from /kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/original_images
Loaded 54000 images from /kaggle/input/datasets/namyajain2810/indian-sign-language/isl_dataset/custom_images
Loading complete.


In [5]:
# ─── Split + Encode ────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np

le = LabelEncoder()
le.fit(sorted(set(orig_labels)))
print("Classes:", len(le.classes_))

X_orig = np.array(orig_data) / 255.0
y_orig = le.transform(orig_labels)

X_cust_full = np.array(cust_data) / 255.0
y_cust_full = le.transform(cust_labels)

# Custom images ka 20% held-out test set
idx_train, idx_test = train_test_split(
    np.arange(len(cust_data)), test_size=0.2,
    stratify=cust_labels, random_state=42
)

cust_train_imgs = [cust_data[i] for i in idx_train]
cust_train_lbls = [cust_labels[i] for i in idx_train]
cust_test_imgs  = [cust_data[i] for i in idx_test]
cust_test_lbls  = [cust_labels[i] for i in idx_test]

# Training pool = original + custom 80%
X_pool = np.array(orig_data + cust_train_imgs) / 255.0
y_pool = le.transform(orig_labels + cust_train_lbls)

# Test set
X_test = np.array(cust_test_imgs) / 255.0
y_test = le.transform(cust_test_lbls)

# Validation split from training pool
X_train, X_val, y_train, y_val = train_test_split(
    X_pool, y_pool, test_size=0.15,
    stratify=y_pool, random_state=42
)

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")

Classes: 36
Train: (45900, 64, 64, 3)
Val:   (8100, 64, 64, 3)
Test:  (10800, 64, 64, 3)


In [6]:
import cv2
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization,
    RandomRotation, RandomZoom, RandomTranslation
)

def build_model(num_classes):
    return Sequential([
        Input(shape=(64, 64, 3)),
        RandomRotation(0.08),
        RandomZoom(0.1),
        RandomTranslation(0.1, 0.1),
        Conv2D(32, (3,3), activation='relu'), BatchNormalization(), MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation='relu'), BatchNormalization(), MaxPooling2D((2,2)),
        Flatten(),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])

model_grey = build_model(len(le.classes_))
model_grey.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Model ready!")
print(model_grey.summary())

I0000 00:00:1782491954.240320    2583 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782491954.245759    2583 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model ready!


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_rotation                 │ (None, 64, 64, 3)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation              │ (None, 64, 64, 3)      │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 62, 62, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 29, 29, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,605,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 36)             │         4,644 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,630,180 (6.22 MB)

 Trainable params: 1,629,988 (6.22 MB)

 Non-trainable params: 192 (768.00 B)

None


In [10]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=4,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "sign_model_grey.keras",
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

history = model_grey.fit(
    X_train, y_train,
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, checkpoint]
)

Epoch 1/30
1434/1435 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9263 - loss: 0.2280
Epoch 1: val_accuracy improved from -inf to 0.96481, saving model to sign_model_grey.keras
1435/1435 ━━━━━━━━━━━━━━━━━━━━ 23s 16ms/step - accuracy: 0.9263 - loss: 0.2280 - val_accuracy: 0.9648 - val_loss: 0.1170
Epoch 2/30
1432/1435 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9305 - loss: 0.2149
Epoch 2: val_accuracy improved from 0.96481 to 0.98037, saving model to sign_model_grey.keras
1435/1435 ━━━━━━━━━━━━━━━━━━━━ 21s 15ms/step - accuracy: 0.9305 - loss: 0.2149 - val_accuracy: 0.9804 - val_loss: 0.0716
Epoch 3/30
1435/1435 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9375 - loss: 0.1953
Epoch 3: val_accuracy did not improve from 0.98037
1435/1435 ━━━━━━━━━━━━━━━━━━━━ 21s 15ms/step - accuracy: 0.9375 - loss: 0.1953 - val_accuracy: 0.9612 - val_loss: 0.0987
Epoch 4/30
1432/1435 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9399 - loss: 0.1903
Epoch 4: val_accuracy did not improve from 0.

In [11]:
# Fast evaluation — batch wise
import numpy as np

predictions = model_grey.predict(X_test, batch_size=256, verbose=1)
pred_classes = np.argmax(predictions, axis=1)
correct = np.sum(pred_classes == y_test)
acc = correct / len(y_test)

print(f"Greyscale model test accuracy: {acc*100:.2f}%")

# Label encoder save karo
import pickle
with open("label_encoder_grey.pkl", "wb") as f:
    pickle.dump(le, f)

print("Label encoder saved!")

43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
Greyscale model test accuracy: 98.52%
Label encoder saved!
